In [16]:
import pandas as pd
cols = ["period", "source_name", "destination_name", "firm", "firm_type",
        "payment instrument", "pickup method",
        "cc1 lcu fee", "cc1 fx margin", "cc1 total cost %", "transparent"]
df = pd.read_excel(
    "rpw_dataset_2011_2025_q3.xlsx",
    sheet_name="Dataset (from Q2 2016)",
    usecols=cols,
)

In [18]:
df.to_csv("rpw_from_2016.csv", index=False)

print(df.shape)
print(df.columns.tolist())



(204469, 11)
['period', 'source_name', 'destination_name', 'firm', 'firm_type', 'payment instrument', 'cc1 lcu fee', 'cc1 fx margin', 'cc1 total cost %', 'transparent', 'pickup method']


In [19]:
latest = df[df["period"] == "2025_3Q"]
us = latest[
    (latest["source_name"] == "United States")
    & (latest["destination_name"].isin(["Mexico", "India", "Philippines"]))
].copy()

print(len(us))
print(us.groupby("destination_name").size())
print(us[["destination_name", "firm", "payment instrument", "pickup method",
          "cc1 lcu fee", "cc1 fx margin", "cc1 total cost %"]].head(20))
print(us.groupby("destination_name")["cc1 total cost %"].agg(["mean", "min", "max"]))

150
destination_name
India          46
Mexico         46
Philippines    58
dtype: int64
       destination_name           firm     payment instrument pickup method  \
203154           Mexico            Ria             Debit card  Bank account   
203186           Mexico       Intermex            Credit card  Bank account   
203187           Mexico       Intermex             Debit card  Bank account   
203218            India           Wise  Bank account transfer  Bank account   
203219            India           Wise            Credit card  Bank account   
203220            India           Wise             Debit card  Bank account   
203222           Mexico           Wise  Bank account transfer  Bank account   
203223           Mexico           Wise            Credit card  Bank account   
203224           Mexico           Wise             Debit card  Bank account   
203231      Philippines           Wise  Bank account transfer  Bank account   
203385            India           Xoom  Ban

In [23]:
import os
import requests
import pandas as pd

def get_cross_rates(date):
    r = requests.get(
        f"https://api.frankfurter.dev/v1/{date}",
        params={"base": "EUR", "symbols": "USD,MXN,INR,PHP"},
        timeout=30,
    )
    r.raise_for_status()
    data = r.json()
    rates = data["rates"]
    return [
        {
            "requested_date": date,
            "rate_date": data["date"],
            "currency": cur,
            "usd_to_local": round(rates[cur] / rates["USD"], 4),
        }
        for cur in ["MXN", "INR", "PHP"]
    ]

rows = []
for d in ["2025-06-02", "2025-06-07"]:
    rows += get_cross_rates(d)

cache = pd.DataFrame(rows)
print(cache)

os.makedirs("data", exist_ok=True)
path = "data/rates_cache.csv"
if os.path.exists(path):
    cache = pd.concat([pd.read_csv(path), cache]).drop_duplicates(
        ["requested_date", "currency"]
    )
cache.to_csv(path, index=False)
print("Saved:", path)

  requested_date   rate_date currency  usd_to_local
0     2025-06-02  2025-06-02      MXN       19.2719
1     2025-06-02  2025-06-02      INR       85.3836
2     2025-06-02  2025-06-02      PHP       55.6555
3     2025-06-07  2025-06-06      MXN       19.1241
4     2025-06-07  2025-06-06      INR       85.7383
5     2025-06-07  2025-06-06      PHP       55.7629
Saved: data/rates_cache.csv


In [25]:
ecb_mxn = 19.2719
fed_mxn = 19.2450	
diff = abs(ecb_mxn - fed_mxn) / fed_mxn * 100
print(f"Difference_mxn: {diff:.3f}%")

ecb_inr = 85.3836
fed_inr = 85.3300	
diff = abs(ecb_inr - fed_inr) / fed_inr * 100
print(f"Difference_inr: {diff:.3f}%")

Difference_mxn: 0.140%
Difference_inr: 0.063%


In [26]:
import os

folders = ["remitcheck", "app", "etl", "data", "docs", "tests"]
for f in folders:
    os.makedirs(f, exist_ok=True)
    open(f"{f}/.gitkeep", "w").close()

print(os.listdir("."))

['.ipynb_checkpoints', 'AdventureWorksDW2022 (1).mdf', 'AdventureWorksDW2022.mdf', 'Anaconda3-2025.12-1-Windows-x86_64.exe', 'app', 'application I-765.pdf', 'Appointment Scheduled (1).pdf', 'Appointment Scheduled (2).pdf', 'Appointment Scheduled.pdf', 'BANL6900-02_Group2_PPR1.docx', 'BANL6900-02_Group2_PPR2.pdf', 'Capstone Project Dataset.zip', 'ChromeSetup (1).exe', 'Claude Setup (1).exe', 'Claude Setup.exe', 'CMS_READMISSIONS ANALYSIS', 'dashboard.mp4', 'data', 'DATAWAREHOUSE.xlsx', 'DataWarehouse_LAB 1_ConceptualModel.docx', 'demo-en.mp3', 'desktop.ini', 'DiscordSetup.exe', 'docs', 'DW_Article_An Analysis of Many-to-Many Relationships Between Fact and Dimensional tables.pdf', 'EAD.jpeg', 'EDA.html', 'etl', 'ETL_Chapter.pdf', 'ETL_Lab_Files', 'Fall 2025_6430_01_Group 1_ProjectFiles', 'For questions in - Port Authority Project 2_Capstone_Tunnels and Bridges Analytics.pbix.zip', 'GoogleMeet_Agent_Installer.msi', 'Google_Meet.msi', 'h2o-3.46.0.9.zip', 'HIPAA_Training_Certificate.pdf', '